# Alerta Chuva SP · Visão Computacional
**GS 2026.2 · CitizenTech · Applied Computer Vision — "Enxergando o mundo real"**

Treinamento customizado de um detector YOLO que reconhece, em fotos e vídeo, **rua alagada**, **veículo ilhado** e **pessoa na água**. O resultado vira um **nível de risco** para o cidadão e alimenta os relatos de alagamento do MVP.

Como rodar: `Ambiente de execução → Alterar o tipo de ambiente → GPU (T4)` e depois `Executar tudo`.

Antes, cadastre sua chave do Roboflow em **🔑 Secrets** (barra lateral esquerda) com o nome `ROBOFLOW_API_KEY` e ative o acesso do notebook. A chave **não** fica no código.

## 1. Ambiente

In [ ]:
!nvidia-smi -L
%pip install -q ultralytics roboflow onnx onnxslim
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())

## 2. Código do projeto
Clona o repositório (troque pela URL de vocês).

In [ ]:
REPO_URL = "https://github.com/zBaldd/GS-VISAO-COMPUTACIONAL.git"  # <- troque
import os
if not os.path.exists("/content/repo"):
    !git clone -q $REPO_URL /content/repo
%cd /content/repo
!ls

## 3. Dados: datasets públicos do Roboflow Universe

Unimos mais de um dataset aberto (licença CC BY 4.0) para ter mais variedade de cenas. Cada um usa nomes de classe diferentes; o script `src/preparar_dataset.py` traduz tudo para as 3 classes do projeto (veja `src/classes.py`) e descarta o resto.

Para trocar ou acrescentar um dataset: abra a página dele no Roboflow Universe → **Download Dataset** → formato **YOLOv8** → **Show download code** e copie `workspace`, `project` e `version` para a lista abaixo.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))

DATASETS = [
    # (workspace, projeto, versão)  -> confira a versão na página do dataset
    ("-alpdg", "flooding-object-detection-opiei", 3),       # flooded_area, flooded_car, flooded_person, flooded_building
    ("ai-course-4kqzb", "flood-detection-aicourse", 1),     # flooded_road, flooded_car, ...
]

pastas = []
for ws, proj, ver in DATASETS:
    try:
        ds = rf.workspace(ws).project(proj).version(ver).download("yolov8", location=f"/content/datasets/{proj}")
        pastas.append(ds.location)
        print("OK", proj, "->", ds.location)
    except Exception as e:
        print("FALHOU", proj, "->", e, "\n   confira workspace/projeto/versão na página do dataset")
# créditos dos datasets (CC BY 4.0 exige atribuição)
import os
os.makedirs("docs", exist_ok=True)
with open("docs/creditos_datasets.md", "w") as f:
    f.write("# Datasets utilizados (Roboflow Universe, CC BY 4.0)

")
    for ws, proj, ver in DATASETS:
        f.write(f"- {proj} v{ver}, autor/workspace `{ws}`: https://universe.roboflow.com/{ws}/{proj}
")
pastas

### 3.1 (Opcional, recomendado) Fotos de ruas secas de SP

Fotos **sem alagamento** ensinam o modelo a não confundir rua molhada, asfalto escuro ou reflexo com enchente (falso positivo). Tire ou junte ~50–150 fotos de ruas de SP e envie um `.zip`. Elas entram sem rótulo (imagens negativas).

In [ ]:
USAR_NEGATIVOS = False   # mude para True se for enviar o zip
NEG = None
if USAR_NEGATIVOS:
    from google.colab import files
    up = files.upload()
    zipname = list(up)[0]
    !rm -rf /content/negativos && mkdir -p /content/negativos && unzip -q -o "$zipname" -d /content/negativos
    NEG = "/content/negativos"
    print(sum(1 for _ in __import__("pathlib").Path(NEG).rglob("*.jp*g")), "fotos negativas")

## 4. Preparar o dataset unificado

In [ ]:
ENTRADAS = " ".join(pastas)
NEG_ARG = f"--negativos {NEG}" if NEG else ""
!python src/preparar_dataset.py --entradas $ENTRADAS --saida /content/dados/alagamento --docs docs $NEG_ARG

In [ ]:
# Amostras com os rótulos (para conferir se a tradução das classes ficou certa)
import random, cv2, matplotlib.pyplot as plt
from pathlib import Path
CL = ["area_alagada", "veiculo_ilhado", "pessoa_na_agua"]; COR = [(30,140,230),(255,165,0),(220,40,40)]
imgs = [p for p in Path("/content/dados/alagamento/train/images").iterdir()]
random.seed(1); amostra = random.sample(imgs, min(8, len(imgs)))
fig, ax = plt.subplots(2, 4, figsize=(18, 8))
for a, p in zip(ax.flat, amostra):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
    lbl = p.parent.parent / "labels" / (p.stem + ".txt")
    for ln in lbl.read_text().splitlines():
        c, cx, cy, bw, bh = ln.split(); c = int(c); cx, cy, bw, bh = map(float, (cx, cy, bw, bh))
        x1, y1, x2, y2 = int((cx-bw/2)*w), int((cy-bh/2)*h), int((cx+bw/2)*w), int((cy+bh/2)*h)
        cv2.rectangle(im, (x1, y1), (x2, y2), COR[c], max(2, w//200)); cv2.putText(im, CL[c], (x1, y1+20), 0, 0.7, COR[c], 2)
    a.imshow(im); a.axis("off")
plt.tight_layout(); plt.savefig("docs/img/amostras_rotuladas.jpg", dpi=90); plt.show()

## 5. Treinamento customizado: YOLO11n × YOLO26n

Partimos de pesos pré-treinados no COCO (*transfer learning*) e fazemos o *fine-tuning* nas nossas 3 classes. Treinamos duas arquiteturas da família YOLO com os mesmos dados e hiperparâmetros e ficamos com a melhor no conjunto de **validação** (o teste só é usado no fim).

Aumentos de dados: mosaico, variação de cor/brilho (chuva, noite, nublado), espelhamento horizontal e escala. Sem espelhamento vertical: a água fica sempre embaixo na cena.

In [ ]:
from ultralytics import YOLO
import json, pandas as pd

HIP = dict(data="/content/dados/alagamento/data.yaml", epochs=80, imgsz=640, batch=16, patience=20,
           seed=42, deterministic=True, cos_lr=True, close_mosaic=10,
           hsv_h=0.015, hsv_s=0.6, hsv_v=0.5, fliplr=0.5, flipud=0.0, scale=0.5, mosaic=1.0,
           project="/content/runs", exist_ok=True, plots=True)

ARQS = ["yolo11n.pt", "yolo26n.pt"]
resultados = {}
for arq in ARQS:
    nome = arq.replace(".pt", "")
    YOLO(arq).train(name=nome, **HIP)
    m = YOLO(f"/content/runs/{nome}/weights/best.pt").val(data=HIP["data"], split="val", imgsz=640, verbose=False)
    resultados[nome] = {"mAP50": round(float(m.box.map50), 4), "mAP50_95": round(float(m.box.map), 4),
                        "precisao": round(float(m.box.mp), 4), "recall": round(float(m.box.mr), 4),
                        "ms_inferencia": round(float(m.speed["inference"]), 2)}
tab = pd.DataFrame(resultados).T; tab

In [ ]:
# Escolhe a melhor arquitetura pela mAP50-95 na VALIDAÇÃO e copia para modelos/best.pt
import shutil, os
melhor = max(resultados, key=lambda k: resultados[k]["mAP50_95"])
os.makedirs("modelos", exist_ok=True); os.makedirs("docs/resultados", exist_ok=True)
shutil.copy2(f"/content/runs/{melhor}/weights/best.pt", "modelos/best.pt")
for f in ["results.png", "labels.jpg", "val_batch0_pred.jpg", "val_batch0_labels.jpg"]:
    src = f"/content/runs/{melhor}/{f}"
    if os.path.exists(src): shutil.copy2(src, f"docs/resultados/treino_{f}")
json.dump({"arquiteturas": resultados, "escolhida": melhor, "hiperparametros":
           {k: v for k, v in HIP.items() if k not in ("project", "exist_ok", "plots")}},
          open("docs/resultados/comparativo.json", "w"), indent=2, ensure_ascii=False)
print("Escolhida:", melhor)

In [ ]:
from IPython.display import Image, display
display(Image(f"/content/runs/{melhor}/results.png", width=1100))

## 6. Avaliação no conjunto de TESTE

Além das métricas clássicas (mAP, precisão, recall), medimos o que importa para o cidadão: **o nível de risco que o sistema mostra está certo?** O nível previsto é comparado com o nível calculado a partir dos rótulos humanos, com a mesma regra de `src/risco.py`.

In [ ]:
!python src/avaliar.py --modelo modelos/best.pt --dados /content/dados/alagamento/data.yaml --saida docs/resultados

In [ ]:
display(Image("docs/resultados/confusion_matrix_normalized.png", width=600))
display(Image("docs/resultados/BoxPR_curve.png", width=600))

## 7. Exemplos de resultado (imagens do teste)

In [ ]:
!python src/detectar.py --modelo modelos/best.pt --fonte /content/dados/alagamento/test/images --salvar /content/pred_teste
import glob
fotos = sorted(glob.glob("/content/pred_teste/*"))[:12]
fig, ax = plt.subplots(3, 4, figsize=(20, 13))
for a, f in zip(ax.flat, fotos):
    a.imshow(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)); a.axis("off")
plt.tight_layout(); plt.savefig("docs/img/exemplos_teste.jpg", dpi=80); plt.show()

## 8. Exportar para o MVP (ONNX → navegador)

In [ ]:
!python src/exportar.py --modelo modelos/best.pt --imgsz 640
!ls -la web/model

## 9. Baixar os resultados

Baixa um zip com `modelos/`, `docs/` e `web/model/`. Descompacte na pasta do repositório no seu computador e faça o commit.

In [ ]:
!zip -q -r /content/resultados_visao.zip modelos docs web/model
from google.colab import files
files.download("/content/resultados_visao.zip")